In [ ]:
## Assignment 11.1 What Drives the Price of a Car?


1. Business Understanding
Goal: To understand what factors make a car more or less expensive. Provide clear recommendations to used car dealership client as to what consumers value in a used car.

If a car is priced too high, inventory will sit on the lot and fewer sales will occur. If a car is priced too low, the dealership loses out on what it could have charged the customer given the features of the car that would make the car of higher value.

Research Framework: CRSIP-DM

Datset: Kaggle used cars dataset with information on 3 million used cars (original dataset). Data subset contains 426k cars. Original data pre-processing done by Berkeley.

Importing Libraries

In [ ]:
import statsmodels.api as sm
from warnings import filterwarnings
filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
from scipy import stats


from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn import set_config
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import SequentialFeatureSelector, SelectFromModel
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
set_config(display="diagram")


2. Data Understanding

Importing Data

In [ ]:
auto = pd.read_csv('/Users/pamelatkrueger/Desktop/Berkeley Certificate/Module 11/Assignment_11.1/practical_application_II_starter/data/vehicles.csv')

In [ ]:
#Look at first few entries of the data
auto.head()

#Note: there are many NaNs upon first glance

In [ ]:
#Understand the variables and basic dataset information
auto.info()

#starting with 426880 entries, 18 variables and the id column has no NaN entries
#First observations about the data:
#We are starting with 426880 entries and 18 variables

In [ ]:
#Look at the count of missing values by variable
auto.isnull().sum().sort_values().plot(kind = 'bar')
plt.show()

In [ ]:
#Get the actual counts of missing data for the features
missing = auto.isnull().sum()
features_missing = missing[missing > 0].to_frame(name = "Missing Value Total")
features_missing

3. Data Preparation

In [ ]:
#drop unhelpful columns
data = auto.drop(columns=['region','id','VIN','model','state'])

#drop if price is missing
data = data.dropna(subset = ['price'])

#Additional note: If we drop rows where there were any missing values, too many observations are lost. Imputation is done later

In [ ]:
#Drop the small number of rows missing odometer or year
data = data.dropna(subset=['odometer', 'year'])

still_missing = data.isnull().sum()
features_still_missing = still_missing[still_missing > 0].to_frame(name = "Still Missing Value Total")
features_still_missing

In [ ]:
#Look at the values the key variables take on:
manufacturer_values = data['manufacturer'].value_counts()
print(manufacturer_values)
condition_values = data['condition'].value_counts()
print(condition_values)
cylinder_values = data['cylinders'].value_counts()
print(cylinder_values)
fuel_values = data['fuel'].value_counts()
print(fuel_values)
title_values = data['title_status'].value_counts()
print(title_values)
transmission_values = data['transmission'].value_counts()
print(transmission_values)
drive_values = data['drive'].value_counts()
print(drive_values)
size_values = data['size'].value_counts()
print(size_values)
type_values = data['type'].value_counts()
print(type_values)
color_values = data['paint_color'].value_counts()
print(color_values)


Exploration of Odometer variable

In [ ]:
#Seaborn boxplot of the odometer variable just to explore
sns.boxplot(data=data, x='odometer').set_title('Boxplot of Odometer')

In [ ]:
#calculate zscores and filter the outlier odometer data out
odometer_z_scores = stats.zscore(data['odometer'])
odometer_abs_z_scores = abs(odometer_z_scores)
data = data[(odometer_abs_z_scores < 3) & (data['odometer'] > 0)]

In [ ]:
#Calculate the values to inspect the odometer variable without outliers
min_odometer = data['odometer'].min()
max_odometer = data['odometer'].max()
mean_odometer = data['odometer'].mean()
std_dev_odometer= data['odometer'].std()
print(f'Min:', min_odometer)
print(f'Max:', max_odometer)
print(f'Mean:', mean_odometer)
print(f'Std Dev:', std_dev_odometer)

Exploration of Price variable

In [ ]:
#Seaborn boxplot of the price variable just to explore
sns.boxplot(data=data, x='price').set_title('Boxplot of Price')

#We see there are a number of outliers that will need to be removed.

In [ ]:
#calculate zscores and filter the outlier price data out
price_z_scores = stats.zscore(data['price'])
price_abs_z_scores = abs(price_z_scores)
data = data[(price_abs_z_scores < 3) & (data['price'] > 50)]

In [ ]:
#Distribution of Price
sns.displot(data['price'], kde = True, bins = 30)
plt.show()

In [ ]:
#Calculate the values to inspect the price variable without outliers
min_price = data['price'].min()
max_price = data['price'].max()
mean_price = data['price'].mean()
std_dev_price = data['price'].std()
print(f'Min:', min_price)
print(f'Max:', max_price)
print(f'Mean:', mean_price)
print(f'Std Dev:', std_dev_price)

In [ ]:
#Log price feature creation
data['log_price'] = np.log(data['price'])

#Examine the new log price feature
sns.boxplot(data=data, x='log_price').set_title('Boxplot of Log of Price')

In [ ]:
#Distribution of Log Price
sns.displot(data['log_price'], kde = True, bins = 50)
plt.show()

In [ ]:
#drop the original price column
data = data.drop(columns=['price'])

Exploration of Year variable

In [ ]:
sns.boxplot(data = data, x = 'year').set_title('Year')

In [ ]:
year_values = data['year'].value_counts()
print(year_values)

In [ ]:
#Calculate the values to inspect the price variable without outliers
min_year = data['year'].min()
max_year= data['year'].max()
mean_year = data['year'].mean()
std_dev_year = data['year'].std()
year_threshold = 3
year_lower_limit = mean_year - (year_threshold * std_dev_year)
year_upper_limit = 2026
print(f'Min:', min_year)
print(f'Max:', max_year)
print(f'Mean:', mean_year)
print(f'Std Dev:', std_dev_year)
print(f'Lower Limit:', year_lower_limit)
print(f'Upper Limit:', year_upper_limit)

In [ ]:
#Remove the year outliers from the data
data = data[(data['year'] >= year_lower_limit) & (data['year'] <= year_upper_limit)]

In [ ]:
sns.boxplot(data = data, x = 'year').set_title('Year')

In [ ]:
#Create a vehicle age variable
data['age'] = pd.Timestamp.now().year - data['year']

In [ ]:
#Drop year
data = data.drop(columns= ['year'])

In [ ]:
#Reset the index now that rows have been filtered out
data = data.reset_index(drop=True)

In [ ]:
#Define X and y and split the dataset in training and testing data
X = data.drop(columns=['log_price'])
y = data['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=42)

print(f'Training set shape:', X_train.shape)
print(f'Test set shape:', X_test.shape)

In [ ]:
#Perform imputation on the training data
numerical_features = ['age', 'odometer']
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

#Replace missing numeric values with median
numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())])

#Replace missing categorical values with mode
categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))])

#Apply transformations
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)])

preprocessor

In [ ]:
#Correlation of every feature
X_train_processed = preprocessor.fit_transform(X_train)

#Get the feature names and define data
feature_names = (numerical_features + list(preprocessor.named_transformers_['cat'].named_steps['encode'].get_feature_names_out(categorical_features)))
corr_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
corr_df['log_price'] = y_train

#Get the correlations
price_corr = corr_df.corr()['log_price'].drop('log_price').sort_values()

#Top and bottom correlations
top_corr = pd.concat([price_corr.head(10), price_corr.tail(10)])

plt.figure(figsize=(8, 8))
plt.barh(top_corr.index, top_corr.values)
plt.title(f'Top 20 features by correlation with log of Price')
plt.xlabel('Correlation with log of price')
plt.show()

4. Modeling

In [ ]:
#Means of getting a list of the feature names
def get_feature_names(fitted_processor):
    return [name.split('__', 1)[1] for name in fitted_processor.get_feature_names_out()]

Starting with standard Linear regression

In [ ]:
#Build a pipeline
linear_pipeline = Pipeline([('preprocessor', preprocessor),
                             ('linear', LinearRegression())])

#Linear regression modeling
linear_pipeline.fit(X_train, y_train)
linear_training_predictions = linear_pipeline.predict(X_train)
linear_test_predictions = linear_pipeline.predict(X_test)

#Compute the errors
linear_training_mse = mean_squared_error(y_train, linear_training_predictions)
linear_test_mse = mean_squared_error(y_test, linear_test_predictions)
linear_training_r2 = r2_score(y_train, linear_training_predictions)
linear_test_r2 = r2_score(y_test, linear_test_predictions)

#Print the coefficients
linear_feature_names = get_feature_names(linear_pipeline.named_steps['preprocessor'])
linear_coefficients = pd.Series(linear_pipeline.named_steps['linear'].coef_, index=linear_feature_names).sort_values()
print('\nStrongest coefficients associated with lower price:')
print(linear_coefficients.head(5))
print('\nStrongest coefficients associated with higher price:')
print(linear_coefficients.tail(5))

print(f'Training MSE:', linear_training_mse)
print(f'Test MSE:', linear_test_mse)
print(f'Training R2:', linear_training_r2)
print(f'Test R2:', linear_test_r2)


Polynomial Regression on Continuous Variables

In [ ]:
#Pipeline to process features and pass in degrees
def polynomial_preprocessor(degree):
    numeric_poly_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('scale', StandardScaler())])
    
    categorical_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)) ])
    
    return ColumnTransformer([
        ('num', numeric_poly_pipeline, numerical_features),
        ('cat', categorical_pipeline, categorical_features) ])

#Pass the degrees to pipeline builder
polynomial_results = []
for degree in range(1, 6):
    polynomial_pipeline = Pipeline([
        ('preprocessor', polynomial_preprocessor(degree)),
        ('linear', LinearRegression())])

    #Fit model and make predictions
    polynomial_pipeline.fit(X_train, y_train)
    train_pred = polynomial_pipeline.predict(X_train)
    test_pred = polynomial_pipeline.predict(X_test)

    #Append the results from each degree model
    polynomial_results.append({
            'degree': degree,
            'train_mse': mean_squared_error(y_train, train_pred),
            'test_mse': mean_squared_error(y_test, test_pred),
            'train_r2': r2_score(y_train, train_pred),
            'test_r2': r2_score(y_test, test_pred)})

polynomial_results_df = pd.DataFrame(polynomial_results)
polynomial_results_df

In [ ]:
plt.figure(figsize=(10, 7))
plt.plot(polynomial_results_df['degree'], polynomial_results_df['train_mse'], label='Training MSE')
plt.plot(polynomial_results_df['degree'], polynomial_results_df['test_mse'], label='Test MSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE of log of price')
plt.title('Training & Test MSE by Polynomial Degree')
plt.legend()
plt.show()

In [ ]:
#Select the degree with the lowest test MSE and assign it as the final model
best_polynomial_degree = int(polynomial_results_df.loc[polynomial_results_df['test_mse'].idxmin(), 'degree'])

polynomial_pipeline = Pipeline([
    ('preprocessor', polynomial_preprocessor(best_polynomial_degree)),
    ('linear', LinearRegression())])

#Fit model and make predictions
polynomial_pipeline.fit(X_train, y_train)
polynomial_training_predictions = polynomial_pipeline.predict(X_train)
polynomial_test_predictions = polynomial_pipeline.predict(X_test)

#Generate error metrics
polynomial_trainingr2 = r2_score(y_train, polynomial_training_predictions)
polynomial_testr2 = r2_score(y_test, polynomial_test_predictions)
polynomial_training_mse = mean_squared_error(y_train, polynomial_training_predictions)
polynomial_test_mse = mean_squared_error(y_test, polynomial_test_predictions)


print(f'Best polynomial degree: {best_polynomial_degree}')
print(f'Training R2:', polynomial_trainingr2)
print(f'Test R2:', polynomial_testr2)
print(f'Training MSE:', polynomial_training_mse)
print(f'Test MSE:', polynomial_test_mse)


Ridge Regression

In [ ]:
#Ridge pipeline
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('ridge', Ridge())])

alphas = {'ridge__alpha': [.001, .01, .05,  1.0, 10.0, 100.0]}
ridge_grid = GridSearchCV(ridge_pipeline, alphas, scoring='neg_mean_squared_error', cv=4)

#Fit model and make predictions
ridge_grid.fit(X_train, y_train)
ridge_best = ridge_grid.best_estimator_
ridge_training_predictions = ridge_best.predict(X_train)
ridge_test_predictions = ridge_best.predict(X_test)

#Generate error metrics
ridge_trainingr2 = r2_score(y_train, ridge_training_predictions)
ridge_testr2 = r2_score(y_test, ridge_test_predictions)
ridge_training_mse = mean_squared_error(y_train, ridge_training_predictions)
ridge_test_mse = mean_squared_error(y_test, ridge_test_predictions)



print('Best alpha:', ridge_grid.best_params_['ridge__alpha'])
print(f'Training R2:', ridge_trainingr2)
print(f'Test R2:', ridge_testr2)
print(f'Training MSE:', ridge_training_mse)
print(f'Test MSE:', ridge_test_mse)


In [ ]:
ridge_cv_results = pd.DataFrame(ridge_grid.cv_results_)

plt.figure(figsize=(10, 5))
plt.plot(ridge_cv_results['param_ridge__alpha'], -ridge_cv_results['mean_test_score'])
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('CMSE')
plt.title('MSE vs Alpha')
plt.show()

Lasso Regression

In [ ]:
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lasso', Lasso(max_iter=5000))])

alphas= {'lasso__alpha': [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1]}
lasso_grid = GridSearchCV(lasso_pipeline, alphas, scoring='neg_mean_squared_error', cv=4)

#Fit the model & get predictions
lasso_grid.fit(X_train, y_train)
lasso_best = lasso_grid.best_estimator_
lasso_training_predictions = lasso_best.predict(X_train)
lasso_test_predictions = lasso_best.predict(X_test)

#Generate error metrics
lasso_training_mse = mean_squared_error(y_train, lasso_training_predictions)
lasso_test_mse = mean_squared_error(y_test, lasso_test_predictions)
lasso_trainingr2 = r2_score(y_train, lasso_training_predictions)
lasso_testr2 = r2_score(y_test, lasso_test_predictions)

print('Best alpha:', lasso_grid.best_params_['lasso__alpha'])
print(f'Training MSE:', lasso_training_mse)
print(f'Test MSE:', lasso_test_mse)
print(f'Training R2:', lasso_trainingr2)
print(f'Test R2:', lasso_testr2)

In [ ]:
#Top coefficients graph
#Get the feature names and coefficients
lasso_feature_names = get_feature_names(lasso_best.named_steps['preprocessor'])
lasso_coefficients = pd.Series(lasso_best.named_steps['lasso'].coef_, index=lasso_feature_names)

#get the top and bottom ten coefficients
top_lasso = pd.concat([lasso_coefficients.sort_values().head(10), lasso_coefficients.sort_values().tail(10)])

plt.figure(figsize=(8, 8))
plt.barh(top_lasso.index, top_lasso.values)
plt.title('Lasso: Top 20 Non-Zero Coefficients')
plt.xlabel('Coefficient (log price scale)')
plt.show()

In [ ]:
results_summary = pd.DataFrame([
    {'Model': 'Linear Regression', 'Train R2': linear_training_r2, 'Test R2': linear_test_r2, 'Train MSE': linear_training_mse, 'Test MSE': linear_test_mse},
    {'Model': f'Polynomial (degree {best_polynomial_degree})', 'Train R2': polynomial_trainingr2, 'Test R2': polynomial_testr2,'Train MSE': polynomial_training_mse, 'Test MSE': polynomial_test_mse},
    {'Model': f'Ridge (alpha={ridge_grid.best_params_["ridge__alpha"]})', 'Train R2': ridge_trainingr2, 'Test R2': ridge_testr2, 'Train MSE': ridge_training_mse, 'Test MSE': ridge_test_mse},
    {'Model': f'Lasso (alpha={lasso_grid.best_params_["lasso__alpha"]})', 'Train R2': lasso_trainingr2, 'Test R2': lasso_testr2, 'Train MSE': lasso_training_mse, 'Test MSE': lasso_test_mse},
]).sort_values('Test MSE').reset_index(drop=True)

results_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_summary['Model'], results_summary['Test R2'])
plt.ylabel('Test R²')
plt.title('Model Comparison Test based on R²')
plt.tight_layout()
plt.show()

Results:

The polynomial degree 5 model had the lowest test MSE so it is the preferable model. The R2 test score means that less than half of the variance of the price is encompassed within the model.

The age of the car and the mileage were highly negatively correlated with the price, as was whether the car was a pick-up, had a transmission type of 'other' (highly positively correlated) or was a sedan (highly negatively correlated).

Based on the regression model, some of the factors driving price up were whether the car was high-end, 12 cylinders, a pickup, a convertible or a manual. Some of the factors driving price down were the title status being missing or parts only, or whether the car was on salvage. Electric and hybrid cars tended to be priced lower as well.

6. Deployment

Note: this section is left here for completeness of the CRISP-DM framework, however it is not applicable in this assignment